# TeleGuard — 01: NSL-KDD Exploration

Quick exploratory look at the raw NSL-KDD training data:
- load with proper column names
- `head()` / `describe()`
- `value_counts()` on the attack label
- visualize the **normal vs anomaly** class imbalance and save the figure to `reports/figures/`.

> Make sure `KDDTrain+.txt` is in `data/raw/` before running.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make the project root importable so we can reuse src/preprocessing.py
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    COLUMN_NAMES,
    LABEL_COL,
    load_raw,
    make_binary_target,
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "KDDTrain+.txt"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
print("Raw path:", RAW_PATH)

Raw path: d:\gan\teleguard\data\raw\KDDTrain+.txt


In [ ]:
import kagglehub
import shutil
from pathlib import Path

# 1. Download the dataset via Kagglehub
source_path = kagglehub.dataset_download("hassan06/nslkdd")
print(f"Downloaded to Kaggle cache: {source_path}")

# 2. Define your target directory (where preprocessing.py is looking)
target_dir = Path(r"D:\gan\teleguard\data\raw")

# Ensure the directory actually exists before copying
target_dir.mkdir(parents=True, exist_ok=True)

# 3. Copy the necessary files from the cache to your target directory
files_to_copy = ["KDDTrain+.txt", "KDDTest+.txt"]

for filename in files_to_copy:
    src_file = Path(source_path) / filename
    dst_file = target_dir / filename
    
    if src_file.exists():
        shutil.copy2(src_file, dst_file)
        print(f"Successfully copied {filename} to {dst_file}")
    else:
        print(f"Warning: {filename} was not found in the downloaded files!")

FileNotFoundError: Raw file not found: d:\gan\teleguard\data\raw\KDDTrain+.txt
Download NSL-KDD and place KDDTrain+.txt / KDDTest+.txt in D:\gan\teleguard\data\raw

In [ ]:
# Summary statistics for the numeric columns
df.describe()

In [ ]:
# How many of each attack type? (multi-class view of the raw label)
df[LABEL_COL].value_counts()

In [ ]:
# Collapse to the binary target: normal (0) vs anomaly (1)
df = make_binary_target(df)
class_counts = df["target"].value_counts().sort_index()
class_counts.index = ["normal (0)", "anomaly (1)"]

total = len(df)
for name, count in class_counts.items():
    print(f"{name:>12}: {count:>8,}  ({100 * count / total:5.2f}%)")

In [ ]:
# Bar chart of the class imbalance -> save to reports/figures/
fig, ax = plt.subplots(figsize=(6, 4))
colors = ["#2a9d8f", "#e76f51"]
bars = ax.bar(class_counts.index, class_counts.values, color=colors)

ax.set_title("NSL-KDD (KDDTrain+) — Class Imbalance")
ax.set_ylabel("Number of records")
ax.bar_label(bars, fmt="{:,.0f}", padding=3)

fig.tight_layout()
out_path = FIG_DIR / "class_imbalance.png"
fig.savefig(out_path, dpi=150)
print("Saved figure ->", out_path)
plt.show()

**Takeaway:** the dataset is imbalanced toward one class, which motivates the GAN-based
augmentation in Phase 2 — we generate synthetic anomalies to balance the training set and
(hopefully) improve rare-anomaly detection.